In [22]:
import os
import pickle
import numpy as np
import plotly.graph_objects as go

folder = "all_pmfs"  
# folder = "all_greedy"
loaded = {}

for fname in os.listdir(folder):
    if fname.endswith(".pkl"):
        varname = fname.replace("_DP_2.pkl", "")
        # varname = fname.replace("_greedy2.pkl", "")
        with open(os.path.join(folder, fname), "rb") as f:
            loaded[varname] = pickle.load(f)

with open('count_mapper.pkl', 'rb') as f:
    counter_mapper = pickle.load(f)

In [37]:
probabilities = np.load('probabilities.npy')

def make_histogram(probs, category):
    fig = go.Figure()


    fig.add_trace(
        go.Bar(
            x=np.arange(len(probs)),   # indices as categories
            y=probs,                   # probabilities
        )
    )

    
    fig.update_layout(
        title=f"{category} PMF",
        xaxis_title="Score",
        yaxis_title="Probability",
        bargap=0.1
    )

    fig.show()

def compute_data_for_PMF(data_dict, category):
    all_pmfs = []
    for key in loaded[category][0].keys():
        if key[1] == 2:
            all_pmfs.append(loaded[category][0][key])

    return all_pmfs

def compute_PMF(category, make_histogram, compute_data_for_PMF, data_dict, probabilities):
    state_pmfs = compute_data_for_PMF(data_dict, category)
    probabilities = probabilities

    complete_pmf = np.zeros(50)

    for i in range(252):
        state = state_pmfs[i]
        for key in state.keys():
            complete_pmf[key] += (probabilities[i] * state[key])


    make_histogram(complete_pmf, category)

    return complete_pmf, probabilities

In [38]:
_,_ = compute_PMF('chance', make_histogram, compute_data_for_PMF, loaded, probabilities)
_,_ = compute_PMF('three_of_a_kind', make_histogram, compute_data_for_PMF, loaded, probabilities)
_,_ = compute_PMF('full_house', make_histogram, compute_data_for_PMF, loaded, probabilities)
_,_ = compute_PMF('fours', make_histogram, compute_data_for_PMF, loaded, probabilities)

In [49]:
DP = np.load('DP_results.npy', allow_pickle=True)
GREEDY = np.load('greedy_results.npy', allow_pickle=True)

data_DP = np.array(DP[0])
data_greedy = np.array(GREEDY[0])

m1, s1 = data_DP.mean(), data_DP.std()
m2, s2 = data_greedy.mean(), data_greedy.std()

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=data_DP,
    opacity=0.6,
    name="DP"
))

fig.add_trace(go.Histogram(
    x=data_greedy,
    opacity=0.6,
    name="Greedy"
))


fig.add_vline(x=m1, line_width=2, line_dash="dash", line_color="blue",
              annotation_text="μ₁", annotation_position="top")
fig.add_vline(x=m1 - s1, line_width=1, line_dash="dot", line_color="blue",
              annotation_text="μ₁ - std", annotation_position="top")
fig.add_vline(x=m1 + s1, line_width=1, line_dash="dot", line_color="blue",
              annotation_text="μ₁ + std", annotation_position="top")

fig.add_vline(x=m2, line_width=2, line_dash="dash", line_color="red",
              annotation_text="μ₂", annotation_position="top")
fig.add_vline(x=m2 - s2, line_width=1, line_dash="dot", line_color="red",
              annotation_text="μ₂ - std", annotation_position="top")
fig.add_vline(x=m2 + s2, line_width=1, line_dash="dot", line_color="red",
              annotation_text="μ₂ + std", annotation_position="top")

fig.update_layout(
    barmode='overlay',  # overlay histograms
    xaxis_title="Yahtzee Score",
    yaxis_title="Count"
)

fig.show()

print(m1, m1 - s2, m1 + s1, m2, m2 - s2, m2 + s2, s1, s2)

206.5823 178.25877515315227 238.43532853277847 114.9845 86.66097515315228 143.30802484684773 31.853028532778477 28.323524846847718


In [ ]:
# probability that a result from data1 belongs to data2, given a value and a sign

def prob_result_belongs_to(data1, data2, value, sign):
    if sign == '>':
        data1_greater = np.sum(data1 > value)
        data2_greater = np.sum(data2 > value)
        return data1_greater / (data1_greater + data2_greater)
    else:
        data1_greater = np.sum(data1 < value)
        data2_greater = np.sum(data2 < value)
        return data1_greater / (data1_greater + data2_greater)

print(prob_result_belongs_to(data_greedy, data_DP, 115, '<'))

0.9964823138557749
